In [ ]:
!pip -q install pypdf sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 70.5 MB/s eta 0:00:00


In [ ]:
import os
import re
import numpy as np
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded file:", pdf_path)

Saving attention_is_all_you_need.pdf to attention_is_all_you_need.pdf
Uploaded file: attention_is_all_you_need.pdf


In [ ]:
reader = PdfReader(pdf_path)

pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()

    if text:
        pages.append({
            "page": page_number,
            "text": text
        })

print("Number of pages:", len(pages))

Number of pages: 15


In [ ]:
for page in pages[:2]:
    print("PAGE:", page["page"])
    print(page["text"][:2000])
    print("\n" + "="*80 + "\n")

PAGE: 1
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirel

In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'-\s+', '', text)
    return text.strip()

for page in pages:
    page["text"] = clean_text(page["text"])

print(pages[0]["text"][:2000])

Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Exper

In [ ]:
CHUNK_SIZE = 500
OVERLAP = 100

chunks = []

for page in pages:

    words = page["text"].split()

    start = 0

    while start < len(words):

        end = start + CHUNK_SIZE

        chunk_words = words[start:end]

        if len(chunk_words) > 30:

            chunk_text = " ".join(chunk_words)

            chunks.append({
                "text": chunk_text,
                "page": page["page"]
            })

        start += CHUNK_SIZE - OVERLAP

print("Total chunks:", len(chunks))

Total chunks: 23


In [ ]:
for i, chunk in enumerate(chunks[:3]):

    print("CHUNK:", i)
    print("PAGE:", chunk["page"])
    print(chunk["text"])
    print("\n" + "="*80 + "\n")

CHUNK: 0
PAGE: 1
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolution

In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (23, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    embeddings.astype("float32")
)

print("Number of vectors stored:", index.ntotal)

Number of vectors stored: 23


In [ ]:
def retrieve_chunks(question, k=5):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    retrieved = []

    for distance, idx in zip(distances[0], indices[0]):

        if idx < len(chunks):

            retrieved.append({
                "text": chunks[idx]["text"],
                "page": chunks[idx]["page"],
                "distance": float(distance)
            })

    return retrieved

In [ ]:
question = "What is the objective of the research paper?"

results = retrieve_chunks(question, k=5)

for i, result in enumerate(results, start=1):

    print(f"\nRESULT {i}")
    print("Page:", result["page"])
    print("Distance:", result["distance"])
    print(result["text"][:1000])


RESULT 1
Page: 12
Distance: 1.6826624870300293
[25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated corpus of english: The penn treebank. Computational linguistics, 19(2):313–330, 1993. [26] David McClosky, Eugene Charniak, and Mark Johnson. Effective self-training for parsing. In Proceedings of the Human Language Technology Conference of the NAACL, Main Conference , pages 152–159. ACL, June 2006. [27] Ankur Parikh, Oscar Täckström, Dipanjan Das, and Jakob Uszkoreit. A decomposable attention model. In Empirical Methods in Natural Language Processing , 2016. [28] Romain Paulus, Caiming Xiong, and Richard Socher. A deep reinforced model for abstractive summarization. arXiv preprint arXiv:1705.04304, 2017. [29] Slav Petrov, Leon Barrett, Romain Thibaux, and Dan Klein. Learning accurate, compact, and interpretable tree annotation. In Proceedings of the 21st International Conference on Computational Linguistics and 44th Annual Meeting of the AC

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("LLM loaded")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded


In [ ]:
def create_context(results):

    context_parts = []

    for i, result in enumerate(results, start=1):

        context_parts.append(
            f"[Source {i} - Page {result['page']}]\n"
            f"{result['text']}"
        )

    return "\n\n".join(context_parts)

In [ ]:
def generate_answer(question, k=5):

    results = retrieve_chunks(question, k)

    context = create_context(results)

    prompt = f"""
Answer the question using ONLY the research paper context provided below.

If the answer cannot be found in the context, say:
"The answer is not available in the uploaded paper."

Do not invent information.

Question:
{question}

Research Paper Context:
{context}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=250
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, results

In [ ]:
question = "What is the objective of the paper?"

answer, sources = generate_answer(question)

print("ANSWER:")
print(answer)

print("\nSOURCES:")

for source in sources:
    print(f"- Page {source['page']}")

ANSWER:
To improve language models

SOURCES:
- Page 13
- Page 12
- Page 9
- Page 15
- Page 5


In [ ]:
def ask_question(question, k=5):

    answer, sources = generate_answer(question, k)

    print("\n" + "="*80)
    print("QUESTION")
    print("="*80)
    print(question)

    print("\n" + "="*80)
    print("ANSWER")
    print("="*80)
    print(answer)

    print("\n" + "="*80)
    print("SOURCES")
    print("="*80)

    pages_used = sorted(set(source["page"] for source in sources))

    for page in pages_used:
        print(f"Research Paper - Page {page}")

    print("="*80)

In [ ]:
questions = [
    "What is the objective of the paper?",
    "What methodology was used?",
    "What datasets were used?",
    "What are the major findings?",
    "What are the limitations?"
]

for question in questions:

    ask_question(question)


QUESTION
What is the objective of the paper?

ANSWER
To improve language models

SOURCES
Research Paper - Page 5
Research Paper - Page 9
Research Paper - Page 12
Research Paper - Page 13
Research Paper - Page 15

QUESTION
What methodology was used?

ANSWER
We used beam search as described in the previous section, but no checkpoint averaging. We present these results in Table 3

SOURCES
Research Paper - Page 7
Research Paper - Page 8
Research Paper - Page 9
Research Paper - Page 12
Research Paper - Page 13

QUESTION
What datasets were used?

ANSWER
English-to-German translation development set, newstest2013, Penn Treebank, Section 22 development set, Section 23 F1 Vinyals & Kaiser el al. (2014) [37] WSJ only, discriminative 88.3 Petrov et al. (2006) [29] WSJ only, discriminative 90.4 Zhu et al. (2013) [40] semi-supervised 92.1 Vinyals & Kaiser el al. (2014) [37] semi-supervised 92.1 Transformer (4 layers) semi-supervised 92.7 Luong et al. (2015) [23] multi-task 93.0 Dyer et al. (2016) 

In [ ]:
while True:

    question = input("\nAsk a question about the research paper (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Exiting...")
        break

    ask_question(question)


Ask a question about the research paper (type 'exit' to stop): what is the main theme of project?

QUESTION
what is the main theme of project?

ANSWER
Transformer, Motivate Self-attention and Discuss its Advantages over Models

SOURCES
Research Paper - Page 2
Research Paper - Page 8
Research Paper - Page 9
Research Paper - Page 13
Research Paper - Page 14

Ask a question about the research paper (type 'exit' to stop): exit
Exiting...
